# Staff Allocation to Projects — Multi-objective MILP

This notebook adapts the modelling workflow of `TP0-Corrigé.ipynb` to the staffing project. It uses SciPy's HiGHS MILP solver (`scipy.optimize.milp`) so it can run without a Gurobi licence. Install dependencies with `pip install numpy scipy pandas matplotlib`.

The model has four objectives: maximize net profit; minimize makespan; minimize employee-project relationships; minimize the number of late projects. Weighted-sum runs generate Pareto candidates. A finite set of weights does **not** guarantee that every Pareto point is found.


## Mathematical formulation

Sets: employees $i\in I$, projects $j\in J$, qualifications $q\in Q$, days $t\in\{1,\ldots,H\}$.

Parameters: $r_{jq}$ = required person-days of skill $q$ for project $j$; $g_j$ = project gain; $d_j$ = due date; $p_j$ = daily late penalty; $a_{iq}=1$ if employee $i$ has qualification $q$; $v_{it}=1$ if employee $i$ is on leave on day $t$.

Binary decisions: $x_{ijqt}=1$ if employee $i$ works on project $j$ with skill $q$ on day $t$; $y_j=1$ if project $j$ is completed; $c_{jt}=1$ if project $j$ completes on day $t$; $u_{ij}=1$ if employee $i$ works on project $j$. Continuous $M$ is the makespan.

**Constraints**

1. Qualification eligibility: create assignment variables only for qualified employees.
2. One assignment per employee/day: $\sum_{j,q}x_{ijqt}\leq 1$.
3. Leave: create no assignment variable on an employee's vacation days.
4. Skill coverage: $\sum_{i,t}x_{ijqt}=r_{jq}y_j$ for every required project-skill pair.
5. Project executed at most once: $\sum_t c_{jt}=y_j$.
6. Assignments occur no later than completion: $x_{ijqt}\leq\sum_{\tau=t}^Hc_{j\tau}$.
7. Link employee-project indicator: $x_{ijqt}\leq u_{ij}$ and $u_{ij}\leq\sum_{q,t}x_{ijqt}$.
8. Makespan: $M\geq t c_{jt}$.

**Objectives**

$f_1=\sum_jg_jy_j-\sum_{j,t>d_j}p_j(t-d_j)c_{jt}$ (maximize net profit).

$f_2=M$ (minimize makespan); $f_3=\sum_{i,j}u_{ij}$ (minimize employee-project relationships); $f_4=\sum_{j,t>d_j}c_{jt}$ (minimize late projects).

Assumption: required person-days can be performed in parallel by different qualified employees; tasks need not be consecutive and have no precedence constraints. A project's completion day is the latest day on which its work is performed.


In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix, csr_matrix

def build_model(data):
    staff, jobs, H = data['staff'], data['jobs'], data['horizon']
    I, J = len(staff), len(jobs)
    # Variables x only exist for qualified employees on non-leave days.
    xkeys = []
    for i, person in enumerate(staff):
        quals, leave = set(person['qualifications']), set(person.get('vacations', []))
        for j, job in enumerate(jobs):
            for q in job['working_days_per_qualification']:
                if q in quals:
                    for t in range(1, H + 1):
                        if t not in leave:
                            xkeys.append((i, j, q, t))
    xidx = {key: k for k, key in enumerate(xkeys)}
    n = len(xkeys)
    yidx = {j: n + j for j in range(J)}
    cidx = {(j,t): n + J + j*H + t-1 for j in range(J) for t in range(1,H+1)}
    ustart = n + J + J*H
    uidx = {(i,j): ustart + i*J + j for i in range(I) for j in range(J)}
    midx = ustart + I*J
    nv = midx + 1
    rows, lows, highs = [], [], []
    def add(co, lo=-np.inf, hi=np.inf):
        rows.append(co); lows.append(lo); highs.append(hi)
    # At most one assignment per employee per day.
    for i in range(I):
        for t in range(1,H+1):
            co = {xidx[k]: 1 for k in xkeys if k[0] == i and k[3] == t}
            if co: add(co, hi=1)
    # Every required skill-day is covered iff the project is selected.
    for j, job in enumerate(jobs):
        for q, req in job['working_days_per_qualification'].items():
            co = {xidx[k]: 1 for k in xkeys if k[1] == j and k[2] == q}
            co[yidx[j]] = -req
            add(co, lo=0, hi=0)
    # Exactly one completion day if selected, none otherwise.
    for j in range(J):
        co = {cidx[(j,t)]: 1 for t in range(1,H+1)}
        co[yidx[j]] = -1
        add(co, lo=0, hi=0)
    # Every assignment must be on or before the chosen completion day.
    for (i,j,q,t), xi in xidx.items():
        co = {xi: 1}
        for tau in range(t,H+1): co[cidx[(j,tau)]] = -1
        add(co, hi=0)
    # Link employee-project indicator to assignments.
    for (i,j,q,t), xi in xidx.items(): add({xi: 1, uidx[(i,j)]: -1}, hi=0)
    for i in range(I):
        for j in range(J):
            co = {uidx[(i,j)]: 1}
            for (ii,jj,q,t), xi in xidx.items():
                if ii == i and jj == j: co[xi] = -1
            add(co, hi=0)
    # Makespan.
    for j in range(J):
        for t in range(1,H+1): add({cidx[(j,t)]: t, midx: -1}, hi=0)
    A = lil_matrix((len(rows), nv), dtype=float)
    for r, co in enumerate(rows):
        for col, value in co.items(): A[r,col] = value
    lb, ub = np.zeros(nv), np.ones(nv)
    ub[midx] = H
    integrality = np.ones(nv, dtype=int); integrality[midx] = 0
    return {'data':data, 'xkeys':xkeys, 'xidx':xidx, 'yidx':yidx, 'cidx':cidx,
            'uidx':uidx, 'midx':midx, 'nv':nv, 'A':csr_matrix(A),
            'clb':np.array(lows), 'cub':np.array(highs), 'lb':lb, 'ub':ub,
            'integrality':integrality}


In [ ]:
def solve_weighted(model, weights=(10,1,1,1), time_limit=30, mip_rel_gap=0.01):
    data, jobs = model['data'], model['data']['jobs']
    H, I, J = data['horizon'], len(data['staff']), len(jobs)
    # Normalize the four objective components before scalarization.
    profit_scale = max(1, sum(j['gain'] for j in jobs))
    diversity_scale, late_scale = max(1,I*J), max(1,J)
    obj = np.zeros(model['nv'])
    for j, job in enumerate(jobs):
        obj[model['yidx'][j]] -= weights[0]*job['gain']/profit_scale
        for t in range(1,H+1):
            if t > job['due_date']:
                ci = model['cidx'][(j,t)]
                obj[ci] += weights[0]*job['daily_penalty']*(t-job['due_date'])/profit_scale
                obj[ci] += weights[3]/late_scale
    obj[model['midx']] = weights[1]/H
    for idx in model['uidx'].values(): obj[idx] = weights[2]/diversity_scale
    result = milp(obj, integrality=model['integrality'],
                  bounds=Bounds(model['lb'],model['ub']),
                  constraints=LinearConstraint(model['A'],model['clb'],model['cub']),
                  options={'time_limit':time_limit,'mip_rel_gap':mip_rel_gap})
    if result.x is None: return None, result
    z = result.x
    selected = [j for j in range(J) if z[model['yidx'][j]] > .5]
    completion, late_penalty, late_count = {}, 0, 0
    for j in range(J):
        for t in range(1,H+1):
            if z[model['cidx'][(j,t)]] > .5:
                completion[j] = t
                if j in selected and t > jobs[j]['due_date']:
                    late_penalty += jobs[j]['daily_penalty']*(t-jobs[j]['due_date'])
                    late_count += 1
    gross = sum(jobs[j]['gain'] for j in selected)
    diversity = sum(z[idx] > .5 for idx in model['uidx'].values())
    makespan = max(completion.values(), default=0)
    solution = {'net_profit':gross-late_penalty, 'gross_profit':gross,
                'late_penalty':late_penalty, 'makespan':makespan,
                'employee_project_pairs':int(diversity), 'late_projects':late_count,
                'completed_projects':[jobs[j]['name'] for j in selected],
                'completion_days':{jobs[j]['name']:t for j,t in completion.items()},
                'status':result.message, 'mip_gap':getattr(result,'mip_gap',None)}
    return solution, result

def dominates(a, b):
    # Profit is maximized; the other three objectives are minimized.
    av = [a['net_profit'], -a['makespan'], -a['employee_project_pairs'], -a['late_projects']]
    bv = [b['net_profit'], -b['makespan'], -b['employee_project_pairs'], -b['late_projects']]
    return all(x >= y for x,y in zip(av,bv)) and any(x > y for x,y in zip(av,bv))

def pareto_filter(solutions):
    unique = {}
    for s in solutions:
        key = (s['net_profit'],s['makespan'],s['employee_project_pairs'],s['late_projects'])
        unique[key] = s
    vals = list(unique.values())
    return [s for s in vals if not any(dominates(other,s) for other in vals if other is not s)]


In [ ]:
from itertools import product

def generate_weights(steps=5):
    return [tuple(v / steps for v in w)
            for w in product(range(steps + 1), repeat=4)
            if sum(w) == steps]

# 56 normalized weight combinations (nonnegative, summing to 1)
weight_sets = generate_weights(5)
print('Normalized weight combinations:', len(weight_sets))

# Weighted runs: useful for discovery, NOT a complete Pareto front.
# Set a reasonable per-run time limit, and inspect solver status/gap.
instances = ['toy_instance.json', 'medium_instance.json', 'large_instance.json']
all_results = {}
for filename in instances:
    with open(filename, encoding='utf-8') as f:
        data = json.load(f)
    model = build_model(data)
    candidates = []
    for weights in weight_sets:
        sol, result = solve_weighted(model, weights, time_limit=30, mip_rel_gap=0)
        if sol is not None:
            sol['weights'] = weights
            sol['proven_optimal'] = result.status == 0
            candidates.append(sol)
    front = pareto_filter(candidates)
    all_results[filename] = front
    print('\n', filename, '| candidate front size:', len(front))
    display(pd.DataFrame([{k:s[k] for k in ['net_profit','makespan','employee_project_pairs','late_projects','proven_optimal']} for s in front]))


## Interpreting the results

A solution is dominated if another solution has at least as much net profit and no worse makespan, employee-project count or late-project count, with at least one strict improvement. Only candidates that are not dominated by another candidate are retained.

The weighted-sum method samples the Pareto front; it can miss unsupported Pareto points. For a complete front, use an epsilon-constraint enumeration (fix bounds on three objectives and optimize the fourth), or a dedicated multi-objective MILP method. For medium and large instances, increase `time_limit` and tighten `mip_rel_gap` to improve optimality certification.

The all-zero schedule may appear as a non-dominated solution because the project brief does not require at least one project to be selected. If the assignment requires a non-empty schedule, add `sum(y_j) >= 1` to the model.


## Exact epsilon-constraint enumeration (integer objectives)

The three minimization objectives are integer-valued. Enumerating all their attainable upper-bound combinations and maximizing profit for each combination can recover **all distinct Pareto objective vectors**, including unsupported points, **if every MILP is solved to proven optimality**. The loop can be extremely large for medium/large instances. Set `max_subproblems=None` to request complete enumeration; a smaller value is a partial run only. If any solve times out or fails, completeness is not certified.


In [ ]:
from scipy.sparse import vstack, csr_matrix

def epsilon_enumeration(model, max_subproblems=None, time_limit_per_solve=None):
    """Enumerate all integer epsilon bounds; exact only when ALL solves certify optimality.

    WARNING: complexity O((H+1)*(I*J+1)*(J+1)) MILPs.
    """
    data = model['data']
    jobs = data['jobs']
    H, I, J = data['horizon'], len(data['staff']), len(jobs)
    n = model['nv']
    # Minimize -profit (including late penalties)
    profit_obj = np.zeros(n)
    late_row = np.zeros(n)
    diversity_row = np.zeros(n)
    makespan_row = np.zeros(n)
    makespan_row[model['midx']] = 1
    for j, job in enumerate(jobs):
        profit_obj[model['yidx'][j]] = -job['gain']
        for t in range(1,H+1):
            if t > job['due_date']:
                ci = model['cidx'][(j,t)]
                profit_obj[ci] = job['daily_penalty']*(t-job['due_date'])
                late_row[ci] = 1
    for idx in model['uidx'].values(): diversity_row[idx] = 1
    extra = csr_matrix(np.stack([makespan_row, diversity_row, late_row]))
    A = vstack([model['A'], extra], format='csr')
    base_lo = np.r_[model['clb'], [-np.inf]*3]
    base_hi = np.r_[model['cub'], [0,0,0]]
    candidates, count, complete = [], 0, True
    for m_bound in range(H+1):
        for d_bound in range(I*J+1):
            for l_bound in range(J+1):
                if max_subproblems is not None and count >= max_subproblems:
                    return pareto_filter(candidates), False, count
                count += 1
                hi = base_hi.copy()
                hi[-3:] = [m_bound, d_bound, l_bound]
                options = {'mip_rel_gap':0.0}
                if time_limit_per_solve is not None:
                    options['time_limit'] = time_limit_per_solve
                res = milp(profit_obj, integrality=model['integrality'],
                           bounds=Bounds(model['lb'], model['ub']),
                           constraints=LinearConstraint(A, base_lo, hi),
                           options=options)
                if res.status == 2:  # certified infeasible
                    continue
                if res.status != 0 or res.x is None:
                    complete = False
                    continue
                z = res.x
                selected = [j for j in range(J) if z[model['yidx'][j]] > .5]
                times = {j:t for j in range(J) for t in range(1,H+1)
                         if z[model['cidx'][(j,t)]] > .5}
                late = sum(t > jobs[j]['due_date'] for j,t in times.items())
                penalty = sum(jobs[j]['daily_penalty']*max(0,t-jobs[j]['due_date'])
                              for j,t in times.items())
                s = {'net_profit':sum(jobs[j]['gain'] for j in selected)-penalty,
                     'makespan':max(times.values(),default=0),
                     'employee_project_pairs':sum(z[idx]>.5 for idx in model['uidx'].values()),
                     'late_projects':late,
                     'completed_projects':[jobs[j]['name'] for j in selected]}
                candidates.append(s)
    return pareto_filter(candidates), complete, count

# Start with a limited test. This is NOT a certified complete front.
with open('toy_instance.json', encoding='utf-8') as f:
    toy = json.load(f)
toy_model = build_model(toy)
toy_front, complete, checked = epsilon_enumeration(toy_model, max_subproblems=50)
print('Subproblems checked:', checked, 'Complete front certified:', complete)
display(pd.DataFrame(toy_front))

# For complete enumeration, remove the subproblem cap:
# toy_front, complete, checked = epsilon_enumeration(toy_model, max_subproblems=None)
# WARNING: medium/large can require a very large number of MILP solves.
